<a href="https://colab.research.google.com/github/Renata-Nascimento/Projeto-U02-CD/blob/main/02_limpeza_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os, re, sys, glob, unicodedata
import numpy as np
import pandas as pd
import requests

base_ibge = pd.read_csv('https://raw.githubusercontent.com/Renata-Nascimento/Projeto-U02-CD/main/saida/base_municipios_2020_2023.csv')
base_sefaz = pd.read_csv('https://raw.githubusercontent.com/Renata-Nascimento/Projeto-U02-CD/main/saida/repasses_2020_2023.csv')

In [2]:
# Extraindo as colunas únicas diretamente do dataframe
municipios = base_ibge[['cod_ibge', 'municipio', 'mesorregiao', 'regiao_intermediaria']].drop_duplicates().reset_index(drop=True)

print(len(municipios), 'municípios')
municipios.head()

167 municípios


,cod_ibge,municipio,mesorregiao,regiao_intermediaria
0,2400109,Acari,Central Potiguar,Caicó
1,2400208,Assú,Oeste Potiguar,Mossoró
2,2400307,Afonso Bezerra,Central Potiguar,Mossoró
3,2400406,Água Nova,Oeste Potiguar,Mossoró
4,2400505,Alexandria,Oeste Potiguar,Mossoró


In [3]:
def chave(nome):
    """'Olho d'Água do Borges' -> 'OLHODAGUADOBORGES'"""
    s = unicodedata.normalize('NFKD', str(nome)).encode('ascii', 'ignore').decode().upper()
    return re.sub(r'[^A-Z]', '', s)

DE_PARA = {
    'ARES': 'AREZ',
    'BOASAUDE': 'JANUARIOCICCO',                 # nome popular (oficial: Januário Cicco)
    'CAICARADORIODOSVENTOS': 'CAICARADORIODOVENTO',
    'GOVDIXSEPTROSADO': 'GOVERNADORDIXSEPTROSADO',
    'IPANGUASSU': 'IPANGUACU',
    'OLHODAGUADOSBORGES': 'OLHODAGUADOBORGES',
    'RUIBARBOSA': 'RUYBARBOSA',
    'SAOMIGUELDETOUROS': 'SAOMIGUELDOGOSTOSO',   # nome antigo
    'SENELOIDESOUZA': 'SENADORELOIDESOUZA',
    'SENGEORGINOAVELINO': 'SENADORGEORGINOAVELINO',
}

COD_POR_CHAVE = dict(zip(municipios['municipio'].map(chave), municipios['cod_ibge']))

In [4]:
base = base_ibge.merge(base_sefaz, on='cod_ibge')
base.head()

,cod_ibge,municipio,mesorregiao,regiao_intermediaria,ano_x,rep_icms,rep_ipi,rep_ipva,rep_royalties,rep_total,...,cad_pessoas,rep_total_real,pib_real_mil,rep_per_capita,pib_per_capita,pct_baixa_renda,porte,ano_y,tipo,valor
0,2400109,Acari,Central Potiguar,Caicó,2020,3.116225e+06,3279.71321,385048.78,44590.552676,3.549144e+06,...,6675,4.393441e+06,163709.383853,414.592918,15448.653756,53.373596,10–20 mil,2020,ICMS,3.116225e+06
1,2400109,Acari,Central Potiguar,Caicó,2020,3.116225e+06,3279.71321,385048.78,44590.552676,3.549144e+06,...,6675,4.393441e+06,163709.383853,414.592918,15448.653756,53.373596,10–20 mil,2020,IPVA,3.850488e+05
2,2400109,Acari,Central Potiguar,Caicó,2020,3.116225e+06,3279.71321,385048.78,44590.552676,3.549144e+06,...,6675,4.393441e+06,163709.383853,414.592918,15448.653756,53.373596,10–20 mil,2020,IPI,3.279713e+03
3,2400109,Acari,Central Potiguar,Caicó,2020,3.116225e+06,3279.71321,385048.78,44590.552676,3.549144e+06,...,6675,4.393441e+06,163709.383853,414.592918,15448.653756,53.373596,10–20 mil,2020,ROYALTIES,4.459055e+04
4,2400109,Acari,Central Potiguar,Caicó,2020,3.116225e+06,3279.71321,385048.78,44590.552676,3.549144e+06,...,6675,4.393441e+06,163709.383853,414.592918,15448.653756,53.373596,10–20 mil,2021,ICMS,3.576256e+06


### Limpeza e Unificação das Bases de Dados (IBGE & SEFAZ)

Vamos selecionar apenas as colunas de interesse de cada tabela:
* **IBGE**: `cod_ibge`, `municipio`, `mesorregiao`, `regiao_intermediaria`, `ano`, `pib_real_mil`, `rep_per_capita`, `pib_per_capita`, `pct_baixa_renda`, `porte`.
* **SEFAZ**: `cod_ibge`, `ano`, `tipo`, `valor`.

In [14]:
import pandas as pd

# 1. Carregar os dados originais do GitHub
raw_ibge = pd.read_csv('https://raw.githubusercontent.com/Renata-Nascimento/Projeto-U02-CD/main/saida/base_municipios_2020_2023.csv')
raw_sefaz = pd.read_csv('https://raw.githubusercontent.com/Renata-Nascimento/Projeto-U02-CD/main/saida/repasses_2020_2023.csv')

# 2. Selecionar apenas as colunas de interesse para a nossa análise
colunas_interesse_ibge = [
    'cod_ibge', 'municipio', 'mesorregiao', 'regiao_intermediaria',
    'ano', 'pib_real_mil', 'rep_per_capita', 'pib_per_capita',
    'pct_baixa_renda', 'porte'
]
colunas_interesse_sefaz = ['cod_ibge', 'ano', 'tipo', 'valor']

df_ibge_clean = raw_ibge[colunas_interesse_ibge]
df_sefaz_clean = raw_sefaz[colunas_interesse_sefaz]

# 3. Mesclar as tabelas baseando-se no Código do IBGE e no Ano correspondente
df_analise = pd.merge(df_ibge_clean, df_sefaz_clean, on=['cod_ibge', 'ano'], how='inner')

# Exibir informações da base consolidada
print(f"Tamanho final da base unificada: {df_analise.shape[0]} linhas e {df_analise.shape[1]} colunas.\n")
display(df_analise.head())

SyntaxError: invalid syntax (1178684435.py, line 13)

### Análise de Qualidade dos Dados (`df_analise`)

Vamos executar um diagnóstico completo para garantir a integridade dos nossos dados antes de iniciar análises mais profundas.

In [11]:
print("=== 1. VERIFICAÇÃO DE VALORES NULOS ===")
missing_data = df_analise.isnull().sum()
display(missing_data[missing_data > 0] if (missing_data > 0).any() else "Nenhum valor nulo encontrado!")

print("\n=== 2. MUNICÍPIOS ÚNICOS POR ANO E TIPO DE REPASSE ===")
# Cada combinação de Ano e Tipo de repasse deve conter exatamente 167 municípios únicos
verificacao_temporal = df_analise.pivot_table(index='ano', columns='tipo', values='cod_ibge', aggfunc='nunique')
display(verificacao_temporal)

print("\n=== 3. RESUMO ESTATÍSTICO DOS VALORES FINANCEIROS ===")
display(df_analise[['pib_real_mil', 'valor']].describe().apply(lambda s: s.apply('{0:.2f}'.format)))

print("\n=== 4. ESTRUTURA E TIPOS DE COLUNA ===")
display(df_analise.dtypes)

=== 1. VERIFICAÇÃO DE VALORES NULOS ===


'Nenhum valor nulo encontrado!'


=== 2. MUNICÍPIOS ÚNICOS POR ANO E TIPO DE REPASSE ===


tipo,ICMS,IPI,IPVA,ROYALTIES
ano,,,,
2020,167,167,167,167
2021,167,167,167,167
2022,167,167,167,167
2023,167,167,167,167



=== 3. RESUMO ESTATÍSTICO DOS VALORES FINANCEIROS ===


,pib_real_mil,valor
count,2672.00,2672.00
mean,569042.16,2996218.98
std,2423132.38,17327459.90
min,26312.54,1481.30
25%,68229.48,26469.15
50%,135549.92,119649.93
75%,325580.50,1819671.64
max,31162043.00,373165581.80



=== 4. ESTRUTURA E TIPOS DE COLUNA ===


,0
cod_ibge,int64
municipio,object
mesorregiao,object
regiao_intermediaria,object
ano,int64
pib_real_mil,float64
rep_per_capita,float64
pib_per_capita,float64
pct_baixa_renda,float64
porte,object


### Formatando a Visualização de Valores Monetários

Vamos usar o styler do pandas para aplicar a formatação de moeda brasileira (`R$`) nas colunas financeiras.

In [12]:
# Criando um mapeamento de formatação para as colunas financeiras
# pib_real_mil está em milhares, então multiplicamos por 1000 se quisermos o valor real total ou formatamos mantendo a escala
formatacao = {
    'pib_real_mil': 'R$ {:,.2f} mil',
    'rep_per_capita': 'R$ {:,.2f}',
    'pib_per_capita': 'R$ {:,.2f}',
    'valor': 'R$ {:,.2f}'
}

# Exibindo as primeiras linhas do DataFrame com a formatação aplicada
df_analise.head().style.format(formatacao, thousands='.', decimal=',')

,cod_ibge,municipio,mesorregiao,regiao_intermediaria,ano,pib_real_mil,rep_per_capita,pib_per_capita,pct_baixa_renda,porte,tipo,valor
0,2.400.109,Acari,Central Potiguar,Caicó,2.020,"R$ 163.709,38 mil","R$ 414,59","R$ 15.448,65","53,373596",10–20 mil,ICMS,"R$ 3.116.225,09"
1,2.400.109,Acari,Central Potiguar,Caicó,2.020,"R$ 163.709,38 mil","R$ 414,59","R$ 15.448,65","53,373596",10–20 mil,IPVA,"R$ 385.048,78"
2,2.400.109,Acari,Central Potiguar,Caicó,2.020,"R$ 163.709,38 mil","R$ 414,59","R$ 15.448,65","53,373596",10–20 mil,IPI,"R$ 3.279,71"
3,2.400.109,Acari,Central Potiguar,Caicó,2.020,"R$ 163.709,38 mil","R$ 414,59","R$ 15.448,65","53,373596",10–20 mil,ROYALTIES,"R$ 44.590,55"
4,2.400.109,Acari,Central Potiguar,Caicó,2.021,"R$ 162.296,79 mil","R$ 440,08","R$ 15.315,35","55,213740",10–20 mil,ICMS,"R$ 3.576.256,10"


In [13]:
# Atualizando o mapeamento de formatação para incluir pct_baixa_renda com o símbolo de porcentagem
formatacao_completa = {
    'pib_real_mil': 'R$ {:,.2f} mil',
    'rep_per_capita': 'R$ {:,.2f}',
    'pib_per_capita': 'R$ {:,.2f}',
    'valor': 'R$ {:,.2f}',
    'pct_baixa_renda': '{:.2f}%'
}

# Exibindo as primeiras linhas do DataFrame com todas as formatações brasileiras aplicadas
df_analise.head().style.format(formatacao_completa, thousands='.', decimal=',')

,cod_ibge,municipio,mesorregiao,regiao_intermediaria,ano,pib_real_mil,rep_per_capita,pib_per_capita,pct_baixa_renda,porte,tipo,valor
0,2.400.109,Acari,Central Potiguar,Caicó,2.020,"R$ 163.709,38 mil","R$ 414,59","R$ 15.448,65","53,37%",10–20 mil,ICMS,"R$ 3.116.225,09"
1,2.400.109,Acari,Central Potiguar,Caicó,2.020,"R$ 163.709,38 mil","R$ 414,59","R$ 15.448,65","53,37%",10–20 mil,IPVA,"R$ 385.048,78"
2,2.400.109,Acari,Central Potiguar,Caicó,2.020,"R$ 163.709,38 mil","R$ 414,59","R$ 15.448,65","53,37%",10–20 mil,IPI,"R$ 3.279,71"
3,2.400.109,Acari,Central Potiguar,Caicó,2.020,"R$ 163.709,38 mil","R$ 414,59","R$ 15.448,65","53,37%",10–20 mil,ROYALTIES,"R$ 44.590,55"
4,2.400.109,Acari,Central Potiguar,Caicó,2.021,"R$ 162.296,79 mil","R$ 440,08","R$ 15.315,35","55,21%",10–20 mil,ICMS,"R$ 3.576.256,10"


In [15]:
# 1. Converter permanentemente as colunas 'cod_ibge' e 'ano' para string (texto) para evitar formatação numérica
df_analise['cod_ibge'] = df_analise['cod_ibge'].astype(str)
df_analise['ano'] = df_analise['ano'].astype(str)

# 2. Definir o mapeamento de formatação apenas para as colunas numéricas de interesse
formatacao_final = {
    'pib_real_mil': 'R$ {:,.2f} mil',
    'rep_per_capita': 'R$ {:,.2f}',
    'pib_per_capita': 'R$ {:,.2f}',
    'valor': 'R$ {:,.2f}',
    'pct_baixa_renda': '{:.2f}%'
}

# 3. Exibir o DataFrame estilizado
# Como as colunas 'cod_ibge' e 'ano' agora são strings, elas não serão afetadas por regras de milhares/decimais
df_analise.head().style.format(formatacao_final, thousands='.', decimal=',')

,cod_ibge,municipio,mesorregiao,regiao_intermediaria,ano,pib_real_mil,rep_per_capita,pib_per_capita,pct_baixa_renda,porte,tipo,valor
0,2400109,Acari,Central Potiguar,Caicó,2020,"R$ 163.709,38 mil","R$ 414,59","R$ 15.448,65","53,37%",10–20 mil,ICMS,"R$ 3.116.225,09"
1,2400109,Acari,Central Potiguar,Caicó,2020,"R$ 163.709,38 mil","R$ 414,59","R$ 15.448,65","53,37%",10–20 mil,IPVA,"R$ 385.048,78"
2,2400109,Acari,Central Potiguar,Caicó,2020,"R$ 163.709,38 mil","R$ 414,59","R$ 15.448,65","53,37%",10–20 mil,IPI,"R$ 3.279,71"
3,2400109,Acari,Central Potiguar,Caicó,2020,"R$ 163.709,38 mil","R$ 414,59","R$ 15.448,65","53,37%",10–20 mil,ROYALTIES,"R$ 44.590,55"
4,2400109,Acari,Central Potiguar,Caicó,2021,"R$ 162.296,79 mil","R$ 440,08","R$ 15.315,35","55,21%",10–20 mil,ICMS,"R$ 3.576.256,10"


In [6]:
ANOS = [2020, 2021, 2022, 2023]   # período da análise
ANO_PRECOS = 2023                # valores em R$ de 2023

MESES = ['JANEIRO', 'FEVEREIRO', 'MARCO', 'ABRIL', 'MAIO', 'JUNHO',
         'JULHO', 'AGOSTO', 'SETEMBRO', 'OUTUBRO', 'NOVEMBRO', 'DEZEMBRO']

arquivos = sorted(glob.glob('dados/sefaz/*.xls'))

def ler_aba(arquivo, aba, ano):
    bruto = pd.read_excel(arquivo, sheet_name=aba, header=None)
    lin_cab = bruto.index[bruto[0].astype(str).str.upper().str.startswith('MUNIC')][0]
    cabecalho = [chave(c) for c in bruto.iloc[lin_cab]]
    corpo = bruto.iloc[lin_cab + 1:]

    k = corpo[0].map(chave).replace(DE_PARA)
    inicio = k[k.isin(list(COD_POR_CHAVE))].index[0]
    fim = corpo.index[(corpo.index > inicio) & corpo[0].astype(str).str.upper().str.contains('TOTA')]
    if len(fim):
        corpo, k = corpo.loc[:fim[0] - 1], k.loc[:fim[0] - 1]
    corpo = corpo.assign(cod_ibge=k.map(COD_POR_CHAVE)).dropna(subset=['cod_ibge'])

    colunas_meses = [j for j, c in enumerate(cabecalho) if c in MESES]
    valores = corpo[colunas_meses].apply(pd.to_numeric, errors='coerce')
    return pd.DataFrame({'cod_ibge': corpo['cod_ibge'].astype(int).values,
                         'ano': ano, 'tipo': aba,
                         'valor': valores.sum(axis=1).values})

partes = []
for arq in arquivos:
    ano = int(re.search(r'(20\d{2})', os.path.basename(arq)).group(1))   # 1º ano que aparece no nome do arquivo
    for aba in ['ICMS', 'IPVA', 'IPI', 'ROYALTIES']:
        partes.append(ler_aba(arq, aba, ano))
repasses = pd.concat(partes, ignore_index=True)
repasses = repasses[repasses['ano'].isin(ANOS)]

# conferência: cada ano × tipo deve ter os 167 municípios
repasses.pivot_table(index='ano', columns='tipo', values='cod_ibge', aggfunc='nunique')

tipo,ICMS,IPI,IPVA,ROYALTIES
ano,,,,
2020,167,167,167,167
2021,167,167,167,167
2022,167,167,167,167
2023,167,167,167,167
